# SCOPe family assignment benchmark

## Description

Using viral subset of the SCOPe dataset, we assess the performance of ESM3Di versus ProstT5 in assigning domains to the correct category.

This is a reproduced version of the identical benchmark used in Foldseek paper, therefore please refer to the paper ([link](https://www.nature.com/articles/s41587-023-01773-0)) for detailed description of the benchmark (Methods - SCOPe benchmark).

---

## SCOPe Dataset preparation

### Download SCOPe40 dataset

In [2]:
%%bash
mkdir -p data/scope
wget --no-check-certificate -q -O data/scope/dir.des.scope.txt \
    "https://scop.berkeley.edu/downloads/parse/dir.des.scope.2.08-stable.txt"
wget --no-check-certificate -q -O data/scope/dir.cla.scope.txt \
    "https://scop.berkeley.edu/downloads/parse/dir.cla.scope.2.08-stable.txt"

### Parse taxonomy information to obtain viral subset of the SCOPe40 dataset

#### Obtain NCBI taxdump files

In [6]:
%%bash
mkdir -p data/ncbi
wget -q -O data/ncbi/taxdump.tar.gz "ftp://ftp.ncbi.nih.gov/pub/taxonomy/taxdump.tar.gz"
tar -xzf data/ncbi/taxdump.tar.gz -C data/ncbi

#### Parse taxonomy files and identify viral entries

In [8]:
%%bash
awk -F'\t' '$7~/scientific/' data/ncbi/names.dmp > data/ncbi/scientific_names.dmp
awk '\
    BEGIN{FS="\t"; OFS="\t"} FNR==1{FC++} \
    FC==1{p[$1]=$3;r[$1]=$5;next} \
    FC==2{n[$1]=$3;next} \
    {x="";while($2>1){x=x?n[$2]","x:n[$2];$2=p[$2]} print $1,x}' \
    data/ncbi/nodes.dmp data/ncbi/scientific_names.dmp \
    <(awk -F'\t|=|,| |]' 'NR==FNR{if($2=="sp")f[$1]=$(NF-1);next} FNR>4 {print $1 "\t" f[$17]}' \
    data/scope/dir.des.scope.txt data/scope/dir.cla.scope.txt) \
    > data/scope/scope40_taxonomy.tsv
awk -F'\t|,' '$2~/Viruses/ {print $1}' data/scope/scope40_taxonomy.tsv > data/scope/scope40_viral_ids.txt
echo "$(wc -l < data/scope/scope40_viral_ids.txt) viral entries identified in the SCOPe40 dataset"

16286 viral entries identified in the SCOPe40 dataset


### Create viral subset of the SCOPe40 dataset

In [10]:
%%bash
mkdir -p data/scope/pdb_viral
cat data/scope/scope40_viral_ids.txt | while read id
do
    echo -e "https://scop.berkeley.edu/downloads/pdbstyle/pdbstyle-2.08/${id:2:2}/${id}.ent\n\tout=data/scope/pdb_viral/${id}.pdb"
done > data/scope/scope40_aria2c.txt
aria2c -i data/scope/scope40_aria2c.txt -x 16 -j 16 --check-certificate=false --quiet=true
foldseek createdb data/scope/pdb_viral data/foldseek/scope40_viral

### Create FASTA files of the viral SCOPe40 dataset

In [ ]:
%%bash
foldseek base:convert2fasta data/foldseek/scope40_viral data/scope/scope40_aa.fa
foldseek base:convert2fasta data/foldseek/scope40_viral_ss data/scope/scope40_3di.fa

---